# Deterministic Optimization and Modeling Group Project

#### Project Partners: Tirdod B., Iñigo, Timothy Cassel
#### Date: 26.11.2024

In [ ]:
# Import libraries

# Optimization libs
from typing import Dict, List, KeysView, Optional, Tuple
from dataclasses import dataclass
from gurobipy import Model, GRB

# Stats / Data libs
import numpy as np
import random
import pandas as pd
from scipy.stats import ttest_ind
from collections import defaultdict

# Graphing libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Additional libs
from itertools import product

ModuleNotFoundError: No module named 'gurobipy'

In [ ]:
# Set seed for reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

# Set global seed for reproducibility
SEED = 42
set_seed(SEED)

### 1. Knapsack Class Implementation
Our first objective is to simply implement the knapsack class we will use as a base to solve this optimization problem. In fact, it is very similar to the one we used in class, and the only key additions are adding the additional neigbourhood constraints, and ensuring that the inputs are correct

In [ ]:
## Original Knasack Class
class InvalidKnapsackError(RuntimeError):
    pass

class Knapsack:
    cost: Dict[int, int]
    votes: Dict[int, int]
    budget: int
    neigbourhoods: int
    proposals: int
    upper_bounds: Dict[int, int]
    lower_bounds: Dict[int, int]

    def __init__(self, cost: Dict[int, int], votes: Dict[int, int], budget: int, neighborhood_assignments: Dict[int, List[int]], impacts: Dict[int, int], lower_bounds: Dict[int, int], upper_bounds: Dict[int, int]):
        if set(cost.keys()) != set(votes.keys()):
            raise InvalidKnapsackError("cost and votess must have the same keys.")

        # Non-negativity constraints
        if budget < 0:
            raise InvalidKnapsackError("Capaity cannot be negative")


        self.cost = cost
        self.votes = votes
        self.budget = budget
        self.neighborhood_assignments = neighborhood_assignments
        self.neighborhoods = len(self.neighborhood_assignments)
        self.total_proposals = sum(len(proposals) for proposals in neighborhood_assignments.values())
        self.impacts = impacts
        self.upper_bounds = upper_bounds
        self.lower_bounds = lower_bounds

        if self.neighborhoods < 0:
            raise InvalidKnapsacKError("Number of neighborhoods cannot be negative")


        # Check that all dictionaries have the same length
        assert len(self.cost) == self.total_proposals
        assert len(self.upper_bounds) == len(self.lower_bounds)
        assert len(self.upper_bounds) == len(self.neighborhood_assignments)

        # Use a set to collect unique proposal indices
        unique_proposals = set()
        for proposals in neighborhood_assignments.values():
            unique_proposals.update(proposals)

        # Get the total count of unique proposals
        num_unique_proposals = len(unique_proposals)

        assert num_unique_proposals == len(self.impacts)

        # Check and print for debugging
        for neighborhood in lower_bounds:
            if neighborhood in upper_bounds:
                lb = lower_bounds[neighborhood]
                ub = upper_bounds[neighborhood]
                assert lb <= ub, (
                    f"Lower bound for neighborhood {neighborhood} exceeds its upper bound."
                )
            else:
                raise KeyError(f"Neighborhood {neighborhood} in lower_bounds is missing in upper_bounds.")


        # Generate a dictionary of all neighboorhoods, proposals and associated impacts.
        self.neighborhood_impacts = {
            k: [(j, self.impacts[j]) for j in self.proposals] for k, self.proposals in self.neighborhood_assignments.items()
        }

    def items(self) -> KeysView:
        return self.votes.keys()

    def get_impacts(self):
        return self.neighborhood_impacts

@dataclass
class KnapsackSolution:
    cost: int
    items: List[int]
    runtime: float
    node_count: int
    iteration_count: int

def solve_mip(kp: Knapsack, seed: int = 42) -> KnapsackSolution:
    """
    Solve the knapsack problem using Gurobi and return the solution with difficulty metrics.

    Parameters:
        kp (Knapsack): The knapsack problem instance.
        seed (int): Seed for deterministic solver behavior.

    Returns:
        KnapsackSolution: The solution and associated metrics.
    """
    m = Model("Knapsack")
    m.setParam('OutputFlag', 0)

    # Set the random seed for deterministic behavior
    m.setParam('Seed', seed)

    # Define binary decision variables for each proposal
    x = m.addVars(kp.cost.keys(), vtype=GRB.BINARY, obj=kp.votes, name="x")

    # Budget constraint
    m.addConstr(sum(kp.cost[j] * x[j] for j in kp.cost.keys()) <= kp.budget, name="BudgetConstraint")

    # Neighborhood constraints
    for k, proposals in kp.neighborhood_assignments.items():
        total_impact = sum(kp.impacts[j] * x[j] for j in proposals)
        m.addConstr(total_impact >= kp.lower_bounds[k], name=f"Neighborhood_{k}_LowerBound")
        m.addConstr(total_impact <= kp.upper_bounds[k], name=f"Neighborhood_{k}_UpperBound")

    # Set optimization objective
    m.ModelSense = GRB.MAXIMIZE
    m.optimize()

    # Check solution status
    if m.Status != GRB.OPTIMAL:
        if m.Status == GRB.INFEASIBLE:
            #print("Model is infeasible. Consider checking constraints or bounds.")
            pass
        elif m.Status == GRB.UNBOUNDED:
            print("Model is unbounded. Consider revising the objective or constraints.")
        else:
            print(f"Optimization ended with status: {m.Status}")
        return KnapsackSolution(cost=0, items=[], runtime=m.Runtime, node_count=m.NodeCount, iteration_count=m.IterCount)

    # Extract solution
    selected_items = [j for j in kp.cost.keys() if x[j].X > 0.5]
    total_cost = sum(kp.cost[j] for j in selected_items)

    return KnapsackSolution(
        cost=total_cost,
        items=selected_items,
        runtime=m.Runtime,
        node_count=m.NodeCount,
        iteration_count=m.IterCount,
    )

## 2. Instance Generation

### 2.1 Simple Custom Instances

#### 2.1.1 Simple custom instance (that should work)
For our first instance, we just want to verify that the class is working as expected. We create an extremely simple problem, that we know should work and does not violate any constraints

In [ ]:
# Define a simple instance
cost = {1: 10, 2: 20, 3: 15, 4: 30}
votes = {1: 40, 2: 60, 3: 50, 4: 70}
budget = 60
neighborhood_assignments = {
    1: [1, 2],
    2: [3, 4]
}
impacts = {1: 20, 2: 25, 3: 10, 4: 30}
lower_bounds = {1: 25, 2: 15}
upper_bounds = {1: 50, 2: 40}

# Create Knapsack instance
knapsack_instance = Knapsack(
    cost=cost,
    votes=votes,
    budget=budget,
    neighborhood_assignments=neighborhood_assignments,
    impacts=impacts,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds
)

# Solve the problem
solution = solve_mip(knapsack_instance)

# Report solution and metrics
print(f"Total Cost: {solution.cost}")
print(f"Selected Items: {solution.items}")
print(f"Runtime: {solution.runtime} seconds")
print(f"Node Count: {solution.node_count}")
print(f"Iteration Count: {solution.iteration_count}")

# Analysis
print(f"Votes for selected items: {sum(votes[item] for item in solution.items)}")
print(f"Impacts per neighborhood:")
for neighborhood, proposals in neighborhood_assignments.items():
    total_impact = sum(impacts[item] for item in solution.items if item in proposals)
    print(f" - Neighborhood {neighborhood}: {total_impact}")


NameError: name 'Model' is not defined

#### 2.1.2 Simple custom instance (That should fail)
For our second instance, we want to just check a instance that should guarantee to fail due to a constraint violation

In [ ]:
# Define a simple instance
cost = {1: 10, 2: 20, 3: 15, 4: 30}
votes = {1: 40, 2: 60, 3: 50, 4: 70}
budget = 60
neighborhood_assignments = {
    1: [1, 2],
    2: [3, 4]
}
impacts = {1: 20, 2: 25, 3: 10, 4: 30}
lower_bounds = {1: 25, 2: 15}
upper_bounds = {1: 50, 2: 20}

# Create Knapsack instance
knapsack_instance = Knapsack(
    cost=cost,
    votes=votes,
    budget=budget,
    neighborhood_assignments=neighborhood_assignments,
    impacts=impacts,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds
)

# Solve the problem
solution = solve_mip(knapsack_instance)
solution

## 2.2 Custom instances with generated data and initial feasibility analysis

### 2.2.0 Implementation of helper functions for analysis, visualization and hyperparameter tuning

#### Methods for analyzing and evaluating instances

In [ ]:
from itertools import combinations

from itertools import combinations
from collections import defaultdict
import numpy as np
from scipy.stats import ttest_ind

def evaluate_instances(instance_metrics: list, group_by_method: bool = False):
    """
    Evaluate multiple instances and perform pairwise statistical comparisons.

    Parameters:
        instance_metrics (list): A list of dictionaries containing instance metrics.
        group_by_method (bool): Whether to group by generation method.

    Returns:
        dict: Evaluation results and pairwise statistical comparisons.
    """
    if group_by_method:
        # Group metrics by generation method
        grouped_metrics = defaultdict(list)
        for metric in instance_metrics:
            grouped_metrics[metric["generation_method"]].append(metric)

        # Individual evaluations for each method
        evaluations = {method: evaluate_instances(metrics, group_by_method=False)
                       for method, metrics in grouped_metrics.items()}

        # Pairwise statistical comparisons
        methods = list(grouped_metrics.keys())
        if len(methods) > 1:
            pairwise_comparisons = {}
            for metric_name in ["runtime", "node_count", "iteration_count"]:
                pairwise_comparisons[metric_name] = {}
                for method1, method2 in combinations(methods, 2):
                    data_1 = [m[metric_name] for m in grouped_metrics[method1] if m["feasible"]]
                    data_2 = [m[metric_name] for m in grouped_metrics[method2] if m["feasible"]]

                    if len(data_1) > 1 and len(data_2) > 1:  # Ensure enough data for comparison
                        t_stat, p_val = ttest_ind(data_1, data_2, equal_var=False)
                        pairwise_comparisons[metric_name][f"{method1} vs {method2}"] = {
                            "T-Statistic": t_stat,
                            "P-Value": p_val,
                        }
                    else:
                        pairwise_comparisons[metric_name][f"{method1} vs {method2}"] = {
                            "T-Statistic": None,
                            "P-Value": None,
                        }

            # Add pairwise comparisons to evaluations
            evaluations["Pairwise Statistical Comparisons"] = pairwise_comparisons

        return evaluations

    # Aggregate metrics for a single group
    runtimes = [m["runtime"] for m in instance_metrics if m["feasible"]]
    node_counts = [m["node_count"] for m in instance_metrics if m["feasible"]]
    iteration_counts = [m["iteration_count"] for m in instance_metrics if m["feasible"]]
    feasibility_rate = sum(1 for m in instance_metrics if m["feasible"]) / len(instance_metrics)

    summary = {
        "Hardness Metrics": {
            "Average Runtime": np.mean(runtimes) if runtimes else None,
            "Average Node Count": np.mean(node_counts) if node_counts else None,
            "Average Iteration Count": np.mean(iteration_counts) if iteration_counts else None,
        },
        "Feasibility Metrics": {
            "Feasibility Rate": feasibility_rate,
        },
    }
    return summary




def display_evaluation_results(evaluation):
    """
    Nicely display the evaluation results in a readable format.

    Parameters:
        evaluation (dict): The evaluation results to display.
    """
    print("=" * 40)
    print("INSTANCE EVALUATION SUMMARY")
    print("=" * 40)

    if isinstance(evaluation, dict):
        for key, value in evaluation.items():
            if isinstance(key, int):
                print(f"\nResults for Generation Method {key}:")
                print("-" * 40)
                display_evaluation_results(value)  # Recursive display for individual methods
            elif key == "Pairwise Statistical Comparisons":
                print("\nPairwise Statistical Comparisons:")
                print("-" * 40)
                for metric, comparisons in value.items():
                    print(f"\nMetric: {metric}")
                    for pair, stats in comparisons.items():
                        t_stat = stats["T-Statistic"]
                        p_val = stats["P-Value"]
                        print(f"  {pair}:")
                        print(f"    T-Statistic: {t_stat:.4f}" if t_stat else "    T-Statistic: None")
                        print(f"    P-Value: {p_val:.4f}" if p_val else "    P-Value: None")
            else:
                if key == "Hardness Metrics":
                    print("\nHardness Metrics:")
                elif key == "Feasibility Metrics":
                    print("\nFeasibility Metrics:")
                print("-" * 40)
                for metric_name, metric_value in value.items():
                    if isinstance(metric_value, (float, int)):
                        print(f"  {metric_name}: {metric_value:.4f}")
                    else:
                        print(f"  {metric_name}: {metric_value}")
    print("=" * 40)



def collect_instance_metrics(kp: Knapsack, solution: KnapsackSolution, generation_method: int) -> Dict:
    """
    Collect metrics for a single instance after solving.

    Parameters:
        kp (Knapsack): The knapsack instance.
        solution (KnapsackSolution): The solution obtained from the solver.
        generation_method (int): Identifier for the instance generation method.

    Returns:
        Dict: Metrics for this instance.
    """
    return {
        "runtime": solution.runtime,
        "node_count": solution.node_count,
        "iteration_count": solution.iteration_count,
        "feasible": bool(solution.items),
        "generation_method": generation_method,
    }

#### Method for plotting Results

In [ ]:
def plot_metrics(instance_metrics: List[Dict], evaluation: Dict):
    """
    Plot metrics from the evaluation results and instance metrics.

    Parameters:
        instance_metrics (List[Dict]): Metrics for individual instances.
        evaluation (Dict): Aggregated evaluation results.
    """
    # Extract metrics
    runtimes = [m["runtime"] for m in instance_metrics if m["feasible"]]
    node_counts = [m["node_count"] for m in instance_metrics if m["feasible"]]
    iteration_counts = [m["iteration_count"] for m in instance_metrics if m["feasible"]]
    feasibility_rates = sum(1 for m in instance_metrics if m["feasible"]) / len(instance_metrics)

    # 1. Runtime Distribution
    plt.figure(figsize=(10, 6))
    plt.hist(runtimes, bins=10, color='skyblue', edgecolor='black', alpha=0.7)
    plt.title("Runtime Distribution", fontsize=14)
    plt.xlabel("Runtime (s)", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.grid(True)
    plt.show()

    # 2. Node Count vs. Iteration Count
    plt.figure(figsize=(10, 6))
    plt.scatter(node_counts, iteration_counts, color='orange', edgecolor='black', alpha=0.7)
    plt.title("Node Count vs. Iteration Count", fontsize=14)
    plt.xlabel("Node Count", fontsize=12)
    plt.ylabel("Iteration Count", fontsize=12)
    plt.grid(True)
    plt.show()

    # 3. Feasibility Rate
    plt.figure(figsize=(6, 6))
    plt.bar(["Feasible", "Infeasible"], [feasibility_rates, 1 - feasibility_rates], color=["green", "red"], alpha=0.7)
    plt.title("Feasibility Rate", fontsize=14)
    plt.ylabel("Proportion", fontsize=12)
    plt.ylim(0, 1)
    plt.grid(axis='y')
    plt.show()

    # 4. Comparative Metrics (if multiple generation methods)
    methods = set(m["generation_method"] for m in instance_metrics)
    if len(methods) > 1:
        method_metrics = {method: [] for method in methods}
        for m in instance_metrics:
            method_metrics[m["generation_method"]].append(m["runtime"])

        avg_runtimes = {method: sum(values) / len(values) for method, values in method_metrics.items()}

        plt.figure(figsize=(10, 6))
        plt.bar(avg_runtimes.keys(), avg_runtimes.values(), color='blue', alpha=0.7)
        plt.title("Average Runtime by Generation Method", fontsize=14)
        plt.xlabel("Generation Method", fontsize=12)
        plt.ylabel("Average Runtime (s)", fontsize=12)
        plt.xticks(list(avg_runtimes.keys()))
        plt.grid(axis='y')
        plt.show()

#### Method for hyperparameter tuning

In [ ]:
def analyze_hyperparameter_impact(instance_metrics: List[Dict], hyperparameters: List[str]) -> pd.DataFrame:
    """
    Analyze the impact of hyperparameters on difficulty metrics, calculating means for feasible instances
    while retaining the proportion of feasible instances.

    Parameters:
        instance_metrics (List[Dict]): List of metrics for all instances.
        hyperparameters (List[str]): List of hyperparameter names to analyze (e.g., "num_proposals").

    Returns:
        pd.DataFrame: Summary of difficulty metrics grouped by hyperparameter combinations.
    """
    # Create a DataFrame for easier analysis
    metrics_df = pd.DataFrame(instance_metrics)

    # Calculate feasibility rate for all instances
    feasibility_rate = metrics_df.groupby(hyperparameters)["feasible"].mean().reset_index()
    feasibility_rate.rename(columns={"feasible": "feasible_mean"}, inplace=True)

    # Filter only feasible instances for metric calculations
    feasible_metrics = metrics_df[metrics_df["feasible"] == 1]

    # Group by hyperparameter combinations and calculate metrics for feasible instances
    grouped = feasible_metrics.groupby(hyperparameters)
    aggregated = grouped.agg({
        "runtime": ["mean", "std"],
        "node_count": ["mean", "std"],
        "iteration_count": ["mean", "std"],
    })

    # Flatten multi-level columns for cleaner output
    aggregated.columns = ['_'.join(col).strip() for col in aggregated.columns.values]
    aggregated.reset_index(inplace=True)

    # Merge feasibility rate into the aggregated metrics
    aggregated = pd.merge(aggregated, feasibility_rate, on=hyperparameters, how="left")

    # Add a hardness score based on feasible instances
    aggregated["hardness_score"] = (
        aggregated["runtime_mean"] +
        aggregated["node_count_mean"] +
        aggregated["iteration_count_mean"]
    )

    return aggregated

#### Method for plotting hyperparameter tuning results

In [ ]:
def plot_hardness_trends(data, y="runtime_mean"):
    """
    Visualize trends in hardness score across different parameters.

    Parameters:
        data (DataFrame): DataFrame containing the parameters and hardness score data.
        y (str): The column name in `data` representing the hardness score (default is "runtime_mean").
        y_scale (str): The scale for the y-axis (default is 'log').
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Impact of Hyperparameters on Hardness Score")

    # 1. Hardness vs num_proposals
    sns.boxplot(ax=axes[0, 0], x="impact_range", y=y, data=data)
    axes[0, 0].set_title("Runtime vs. Impact Range")
    axes[0, 0].set_xlabel("Impact Range")
    axes[0, 0].set_ylabel("Runtime Mean")

    # 2. Hardness vs num_neighborhoods
    sns.boxplot(ax=axes[0, 1], x="num_neighborhoods", y=y, data=data)
    axes[0, 1].set_title("Runtime vs. Number of Neighborhoods")
    axes[0, 1].set_xlabel("Number of Neighborhoods")
    axes[0, 0].set_ylabel("Runtime Mean")

    # 3. Hardness vs budget_factor
    sns.boxplot(ax=axes[1, 0], x="budget_factor", y=y, data=data)
    axes[1, 0].set_title("Runtime vs. Budget Factor")
    axes[1, 0].set_xlabel("Budget Factor")
    axes[0, 0].set_ylabel("Runtime Mean")

    # 4. Hardness vs bound_tightness
    sns.boxplot(ax=axes[1, 1], x="bound_tightness", y=y, data=data)
    axes[1, 1].set_title("Runtime vs. Bound Tightness")
    axes[1, 1].set_xlabel("Bound Tightness")
    axes[0, 0].set_ylabel("Runtime Mean")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
def plot_feasibility_trends(data, y="feasible_mean"):
    """
    Visualize trends in hardness score across different parameters.

    Parameters:
        data (DataFrame): DataFrame containing the parameters and hardness score data.
        y (str): The column name in `data` representing the hardness score (default is "feasible_mean").
        y_scale (str): The scale for the y-axis (default is 'log').
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Impact of Hyperparameters on Hardness Score")

    # 1. Hardness vs num_proposals
    sns.boxplot(ax=axes[0, 0], x="impact_range", y=y, data=data)
    axes[0, 0].set_title("Feasibility vs. Impact Range")
    axes[0, 0].set_xlabel("Impact Range")
    axes[0, 0].set_ylabel("Feasible Mean")

    # 2. Hardness vs num_neighborhoods
    sns.boxplot(ax=axes[0, 1], x="num_neighborhoods", y=y, data=data)
    axes[0, 1].set_title("Feasibility vs. Number of Neighborhoods")
    axes[0, 1].set_xlabel("Number of Neighborhoods")
    axes[0, 1].set_ylabel("Feasible Mean")

    # 3. Hardness vs budget_factor
    sns.boxplot(ax=axes[1, 0], x="budget_factor", y=y, data=data)
    axes[1, 0].set_title("Feasibility vs. Budget Factor")
    axes[1, 0].set_xlabel("Budget Factor")
    axes[1, 0].set_ylabel("Feasible Mean")

    # 4. Hardness vs bound_tightness
    sns.boxplot(ax=axes[1, 1], x="bound_tightness", y=y, data=data)
    axes[1, 1].set_title("Feasibility vs. Bound Tightness")
    axes[1, 1].set_xlabel("Bound Tightness")
    axes[1, 1].set_ylabel("Feasible Mean")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

### 2.2.1 - Uniform Distribution Instance Generation Method

##### 2.2.2.1 Define the method

In [ ]:
def generate_instance(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    vote_range: Tuple[int, int],
    impact_range: Tuple[int, int],
    bound_tightness: float = 0.8,
    seed: int = None,
) -> Knapsack:
    """
    Generate a random instance of the knapsack problem.

    Parameters:
        num_proposals (int): Total number of proposals.
        num_neighborhoods (int): Number of neighborhoods.
        budget (int): Budget for the instance.
        cost_range (Tuple[int, int]): Min and max range for proposal costs.
        vote_range (Tuple[int, int]): Min and max range for proposal votes.
        impact_range (Tuple[int, int]): Min and max range for impacts.
        bound_tightness (float): Fraction of total neighborhood impact used for bounds.
        seed (int, optional): Seed for reproducibility. Defaults to None.

    Returns:
        Knapsack: A randomly generated knapsack instance.
    """
    # Set seed for reproducibility
    if seed is not None:
        random.seed(seed)

    # Generate costs, votes, and impacts uniformly
    costs = {i: random.randint(*cost_range) for i in range(1, num_proposals + 1)}
    votes = {i: random.randint(*vote_range) for i in range(1, num_proposals + 1)}
    impacts = {i: random.randint(*impact_range) for i in range(1, num_proposals + 1)}

    # Assign proposals to neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    for proposal in range(1, num_proposals + 1):
        assigned_neighborhood = random.randint(1, num_neighborhoods)
        neighborhood_assignments[assigned_neighborhood].append(proposal)

    # Generate neighborhood bounds
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * 0.5)  # Lower bound
        upper_bounds[k] = int(bound_tightness * total_impact * 1.5)  # Upper bound

    # Create and return the knapsack instance
    return Knapsack(
        cost=costs,
        votes=votes,
        budget=budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )

##### 2.2.1.2 Run instances

In [ ]:
instances = []
for i in range(1):
    instance = generate_instance(
        num_proposals=3000,
        num_neighborhoods=50,
        budget=20000,
        cost_range=(10, 50),
        vote_range=(20, 100),
        impact_range=(5, 15),
        bound_tightness=0.8,
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_1 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics_1 = collect_instance_metrics(instance, solution, generation_method=1)
    results_1.append(metrics_1)

# Evaluate and compare results grouped by method
evaluation_1 = evaluate_instances(results_1, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_1)

##### 2.2.1.3 Hyperparameter tuning

In [ ]:
metrics = []

# Define hyperparameter combinations
parameter_space = {
    "num_proposals": [3000],
    "num_neighborhoods": [13, 27, 45, 46, 49, 53, 64, 89, 98, 143, 212], # 22, 23, 25, 30, 50, 100
    "budget_factor": [0.33], #0.1, 0.2, 0.29, 0.3, 0.31, 0.32, 0.34, 0.35
    "cost_range": [(20, 100)], #(10, 500), (500,1000), 100, 500), (20, 200)
    "vote_range": [(10,100)],
    "impact_range": [(10, 100)], # (5, 15),
    "bound_tightness": [0.18] # 0.1, 0.3, 0.4, 0.14, 0.16,
}

# Create all combinations of hyperparameters
parameter_combinations = list(product(*parameter_space.values()))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for multiple runs per configuration
        instance = generate_instance(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            vote_range=param_dict["vote_range"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            seed=42 + i,
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=1)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)

##### 2.2.1.4 Hyperparameter results

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget_factor", "cost_range", "bound_tightness", "impact_range"]
hardness_analysis_df_1 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
hardness_analysis_df_1.sort_values(by='runtime_mean', ascending = False).head(25)

##### 2.2.1.5 Plot

In [ ]:
# Assuming `hardness_analysis_df` is your DataFrame
plot_hardness_trends(data=hardness_analysis_df_1)

In [ ]:
budget = 0.33 * 60 * 3000

# Running again for the winner (n=30)
instances = []
for i in range(100):
    instance = generate_instance(
        num_proposals=3000,
        num_neighborhoods=13,
        budget=budget,
        cost_range=(20, 100),
        vote_range=(10, 100),
        impact_range=(10, 100),
        bound_tightness=0.18,
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_1 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics_1 = collect_instance_metrics(instance, solution, generation_method=1)
    results_1.append(metrics_1)

# Evaluate and compare results grouped by method
evaluation_1 = evaluate_instances(results_1, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_1)

NameError: name 'generate_instance' is not defined

In [ ]:
# Plot results of this instance generation for the 50 instances.
plot_metrics(results_1, evaluation_1)

#### Outcome:
For the uniform instance generation method, we found that setting n=3000 generates instances with sufficient difficulty as a baseline (~0.75 seconds). So for the remainder of the project, we will stick with n=3000 proposals, and vary the remaining parameters.

## 2.2.2 Exponential and bi-modal distributions for votes and cost

Does the distribution affect the difficulty? I.e. do non-uniform distributions make the instances harder to solve?

##### 2.2.2.1 Define the method

In [ ]:
def generate_instance_non_uniform(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    vote_range: Tuple[int, int],
    impact_range: Tuple[int, int],
    bound_tightness: float = 0.8,
    seed: int = None,
) -> Knapsack:
    """
    Generate a random instance with non-uniform distributions for costs and votes.

    Parameters:
        num_proposals (int): Total number of proposals.
        num_neighborhoods (int): Number of neighborhoods.
        budget (int): Budget for the instance.
        cost_range (Tuple[int, int]): Min and max range for proposal costs.
        vote_range (Tuple[int, int]): Min and max range for proposal votes.
        impact_range (Tuple[int, int]): Min and max range for impacts.
        bound_tightness (float): Fraction of total neighborhood impact used for bounds.
        seed (int, optional): Seed for reproducibility. Defaults to None.

    Returns:
        Knapsack: A randomly generated knapsack instance.
    """
    if seed is not None:
        np.random.seed(seed)

    # Generate costs with an exponential distribution adjusted to fit cost_range
    costs = {i: max(cost_range[0], min(cost_range[1], int(np.random.exponential(scale=(cost_range[1] - cost_range[0]) / 2))))
             for i in range(1, num_proposals + 1)}

    # Generate votes with a bimodal distribution adjusted to fit vote_range
    mid_vote = (vote_range[0] + vote_range[1]) / 2
    votes = {}
    for i in range(1, num_proposals + 1):
        if np.random.rand() < 0.5:
            votes[i] = max(vote_range[0], min(vote_range[1], int(np.random.normal(loc=mid_vote - 20, scale=10))))
        else:
            votes[i] = max(vote_range[0], min(vote_range[1], int(np.random.normal(loc=mid_vote + 20, scale=10))))

    # Generate impacts uniformly
    impacts = {i: np.random.randint(*impact_range) for i in range(1, num_proposals + 1)}

    # Assign proposals to neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    for proposal in range(1, num_proposals + 1):
        assigned_neighborhood = np.random.randint(1, num_neighborhoods + 1)
        neighborhood_assignments[assigned_neighborhood].append(proposal)

    # Generate neighborhood bounds
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * 0.5)
        upper_bounds[k] = int(bound_tightness * total_impact * 1.5)

    # Create and return the knapsack instance
    return Knapsack(
        cost=costs,
        votes=votes,
        budget=budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )


##### 2.2.2.2 Generate initial instances

In [ ]:
# Evaluation
instances = []
for i in range(1):
    instance = generate_instance_non_uniform(
        num_proposals=3000,
        num_neighborhoods=10,  # Proposals split across 10 neighborhoods
        budget=20000,  # Budget sufficient for about 20-30% of proposals
        cost_range=(10, 50),  # Smaller costs
        vote_range=(20, 100),  # Votes proportional to costs
        impact_range=(10, 100),  # Modest impact ranges
        bound_tightness=0.25,  # Reasonable tightness for feasibility
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_2 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=2)
    results_2.append(metrics)

# Evaluate and compare results grouped by method
evaluation_2 = evaluate_instances(results_2, group_by_method=True)

# Display grouped evaluation results
display_evaluation_results(evaluation_2)

#### 2.2.2.3 Tune Hyperparameters

In [ ]:
metrics = []

# Define hyperparameter combinations
parameter_space = {
    "num_proposals": [3000],
    "num_neighborhoods": [10, 15, 50, 100], # 10, 11, 12, 13, 14, 16, 17, 18
    "budget_factor": [0.25], # 0.3, 0.35, 0.2,  0.26, 0.27
    "cost_range": [(10, 50)], #(10, 500), (500,1000), 100, 500), (20, 200)
    "vote_range": [(20,100)],
    "impact_range": [(10, 100)], # (5, 15),
    "bound_tightness": [0.20, 0.23, 0.25, 0.27] # 0.1, 0.3, 0.4
}

# Create all combinations of hyperparameters
parameter_combinations = list(product(*parameter_space.values()))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for multiple runs per configuration
        instance = generate_instance_non_uniform(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            vote_range=param_dict["vote_range"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            seed=42 + i,
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=2)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)

##### 2.2.2.4 Analyze hyperparameters (adjust tuning) -> choose hardest parameters and select winner

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_proposals", "num_neighborhoods", "budget_factor", "cost_range", "bound_tightness", "impact_range"]
hardness_analysis_df_2 = analyze_hyperparameter_impact(metrics, hyperparameters)

# Display the DataFrame
hardness_analysis_df_2.sort_values(by='runtime_mean', ascending = False).head(25)

##### 2.2.2.5 Plot hyperparameters

In [ ]:
# Assuming `hardness_analysis_df` is your DataFrame
plot_hardness_trends(data=hardness_analysis_df_2)

In [ ]:
# Run the winner again
budget = 3000 * 30 * 0.25

# Evaluation
instances = []
for i in range(100):
    instance = generate_instance_non_uniform(
        num_proposals=3000,
        num_neighborhoods=15,
        budget=budget,
        cost_range=(10, 50),
        vote_range=(20, 100),
        impact_range=(10, 100),
        bound_tightness=0.25,
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_2 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics_2 = collect_instance_metrics(instance, solution, generation_method=2)
    results_2.append(metrics_2)

# Evaluate and compare results grouped by method
evaluation_2 = evaluate_instances(results_2, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_2)

In [ ]:
# Plot results of this instance generation for the 50 instances.
plot_metrics(results_2, evaluation_2)

## 2.2.3 Correlated Costs and correlated impacts

#### 2.2.3.1 Define the method: Correlated Costs to votes, and correlated costs to impacts

In [ ]:
def generate_instance_correlated(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    impact_range: Tuple[int, int],
    bound_tightness: float,
    seed: int = None,
) -> Knapsack:
    """
    Generate a random instance of the knapsack problem with correlated parameters.

    Parameters:
        num_proposals (int): Total number of proposals.
        num_neighborhoods (int): Number of neighborhoods.
        budget (int): Budget for the instance.
        cost_range (Tuple[int, int]): Min and max range for proposal costs.
        impact_range (Tuple[int, int]): Min and max range for impacts.
        bound_tightness (float): Fraction of total neighborhood impact used for bounds.
        seed (int, optional): Seed for reproducibility. Defaults to None.

    Returns:
        Knapsack: A randomly generated knapsack instance.
    """
    # Set seed for reproducibility
    if seed is not None:
        random.seed(seed)

    # Generate costs
    costs = {i: random.randint(*cost_range) for i in range(1, num_proposals + 1)}

    # Generate votes correlated to costs
    votes = {i: max(1, int(costs[i] * random.uniform(1.2, 1.8))) for i in range(1, num_proposals + 1)}

    # Generate impacts correlated to votes
    impacts = {i: max(impact_range[0], min(impact_range[1], int(votes[i] * random.uniform(0.5, 0.8))))
               for i in range(1, num_proposals + 1)}

    # Assign proposals to neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    for proposal in range(1, num_proposals + 1):
        assigned_neighborhood = random.randint(1, num_neighborhoods)
        neighborhood_assignments[assigned_neighborhood].append(proposal)

    # Generate neighborhood bounds
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * 0.5)  # Lower bound
        upper_bounds[k] = int(bound_tightness * total_impact * 1.5)  # Upper bound

    # Create and return the knapsack instance
    return Knapsack(
        cost=costs,
        votes=votes,
        budget=budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )


##### 2.2.3.2 Analyze instance

In [ ]:
# Evaluation
instances = []
for i in range(2):
    instance = generate_instance_correlated(
        num_proposals=3000,
        num_neighborhoods=25,  # Proposals split across 10 neighborhoods
        budget= 36000,
        cost_range=(20, 100),
        impact_range=(10, 100),
        bound_tightness=0.5,  # 0.5 creates a harder instance, but makes it quite infeasible (almost 95% of times)
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_3 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=3)
    results_3.append(metrics)

# Evaluate and compare results grouped by method
evaluation_3 = evaluate_instances(results_3, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_3)

##### 2.2.3.3 Tune Hyperparameters

In [ ]:
metrics = []

# Define hyperparameter combinations
parameter_space = {
    "num_proposals": [3000],
    "num_neighborhoods": [25, 27], # [10, 50, 100, 24, 26
    "budget_factor": [0.2, 0.4], # 0.19, 0.21
    "cost_range": [(20, 100)], #(10, 500), (500,1000), 100, 500), (20, 200)
    "impact_range": [(10, 100)], # (5, 15),
    "bound_tightness": [0.48, 0.52] # 0.1, 0.3, 0.4, 0.15, 0.2, 0.25,
}

# Create all combinations of hyperparameters
parameter_combinations = sorted(list(product(*parameter_space.values())))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(3):  # Repeat for 3 iterations
        instance = generate_instance_correlated(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            seed=SEED + i
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=3)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)


##### 2.2.3.4 Analyze Hyperparameters

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget_factor", "cost_range", "bound_tightness", "impact_range"]
hardness_analysis_df_3 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
hardness_analysis_df_3.sort_values(by='runtime_mean', ascending = False).head(25)

##### 2.2.3.5 Plot Hyperparamters

In [ ]:
# Plot the hardness of the hyperparameters
plot_hardness_trends(data=hardness_analysis_df_3)

##### Winner Method 3

In [ ]:
# Evaluation
budget = 60 * 3000 * 0.2

instances = []
for i in range(150):
    # Skip iteration 118
    if i == 117:
        print(f"Skipping iteration {i}")
        continue
    instance = generate_instance_correlated(
        num_proposals=3000,
        num_neighborhoods=25,
        budget= budget,
        cost_range=(20, 100),
        impact_range=(10, 100),
        bound_tightness=0.5,
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

counter = 1
results_3 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=3)
    results_3.append(metrics)
    print(f"completed iteration {counter} " )
    counter += 1

# Evaluate and compare results grouped by method
evaluation_3 = evaluate_instances(results_3, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_3)

In [ ]:
# Plot results of this instance generation for the 30 instances.
plot_metrics(results_3, evaluation_3)

## 2.2.4 Instance Generation Method: Strongly correlated Costs

##### 2.2.4.1 Define generation method

In [ ]:
def generate_instance_strongly_correlated(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    noise_level: float,
    bound_tightness: float,
    seed: int = None,
) -> Knapsack:
    if seed is not None:
        random.seed(seed)

    # Generate costs
    costs = {i: random.randint(*cost_range) for i in range(1, num_proposals + 1)}

    # Generate votes strongly correlated with costs + noise
    votes = {i: max(1, int(costs[i] * random.uniform(1.5 - noise_level, 1.5 + noise_level)))
             for i in range(1, num_proposals + 1)}

    # Generate impacts proportional to votes
    impacts = {i: max(1, int(votes[i] * random.uniform(0.4, 0.6)))
               for i in range(1, num_proposals + 1)}

    # Assign proposals to neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    for proposal in range(1, num_proposals + 1):
        assigned_neighborhood = random.randint(1, num_neighborhoods)
        neighborhood_assignments[assigned_neighborhood].append(proposal)

    # Generate neighborhood bounds
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * 0.5)  # Lower bound
        upper_bounds[k] = int(bound_tightness * total_impact * 1.5)  # Upper bound


    return Knapsack(
        cost=costs,
        votes=votes,
        budget=budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )


##### 2.2.4.2 Run initial instances

In [ ]:
instances = []
for i in range(1):
    instance = generate_instance_strongly_correlated(
        num_proposals= 3000,
        num_neighborhoods=25,
        budget=36000,
        cost_range=(20, 100),
        noise_level=4,
        bound_tightness=0.2,
        seed=42+i,
    )
    instances.append(instance)

results_4 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=4)
    results_4.append(metrics)


evaluation_4 = evaluate_instances(results_4, group_by_method=False)

# Display evaluation results
display_evaluation_results(evaluation_4)

NameError: name 'Model' is not defined

##### 2.2.4.3 Hyperparameter tuning

In [ ]:
metrics = []

# Define hyperparameter combinations (Adapt the grid while finding harder combinations) - Record combinations tried as comments
parameter_space = {
    "num_proposals": [3000],
    "num_neighborhoods": [29, 30], # [10, 25, 50, 100, 27, 28, 29,
    "budget_factor": [0.2, 0.25], # 0.1,0.2, 0.3
    "cost_range": [(20, 100)], #(20, 100), (10,50)
    "bound_tightness": [0.2, 0.24], # 0.1, 0,2, 0.3, 0.4, 0.15, 0.25,
    "noise_level": [3.6, 3.7], #2,3
}

# Create all combinations of hyperparameters
parameter_combinations = sorted(list(product(*parameter_space.values())))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for 1 iterations for each combination
        instance = generate_instance_strongly_correlated(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            bound_tightness=param_dict["bound_tightness"],
            noise_level=param_dict["noise_level"],
            seed=SEED + i
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=4)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)


##### 2.2.4.4 Analyze hyperparameters

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget_factor", "cost_range", "bound_tightness", "noise_level"]
hardness_analysis_df_4 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
hardness_analysis_df_4.sort_values(by='runtime_mean', ascending = False).head(15)

##### Winner Run

In [ ]:
budget = 3000 * 0.2 * 60

instances = []
for i in range(30):
    instance = generate_instance_strongly_correlated(
        num_proposals= 3000,
        num_neighborhoods=29,
        budget=budget,
        cost_range=(20, 100),
        noise_level=3.6,
        bound_tightness=0.2,
        seed=42+i,
    )
    instances.append(instance)

counter = 1
results_4 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=4)
    results_4.append(metrics)
    print(f"completed iteration {counter} " )
    counter += 1

evaluation_4 = evaluate_instances(results_4, group_by_method=False)

# Display evaluation results
display_evaluation_results(evaluation_4)

##### 2.2.4.6 Plot Evaluation

In [ ]:
# Plot results of this instance generation for the 30 instances.
plot_metrics(results_4, evaluation_4)

## 2.2.5 Instance Generation Method: Multi-tiered Neighborhoods

##### 2.2.5.1 Define generation method

In [ ]:
def generate_instance_multi_tiered(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    vote_range: Tuple[int, int],
    impact_range: Tuple[int, int],
    bound_tightness: float = 0.8,
    max_children: int = 3,  # Max number of children per neighborhood
    seed: int = None,
) -> Knapsack:
    if seed is not None:
        random.seed(seed)

    # Generate costs, votes, and impacts
    costs = {i: random.randint(*cost_range) for i in range(1, num_proposals + 1)}
    votes = {i: random.randint(*vote_range) for i in range(1, num_proposals + 1)}
    impacts = {i: random.randint(*impact_range) for i in range(1, num_proposals + 1)}

    # Assign proposals to neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    for proposal in range(1, num_proposals + 1):
        assigned_neighborhood = random.randint(1, num_neighborhoods)
        neighborhood_assignments[assigned_neighborhood].append(proposal)

    # Define hierarchical structure
    neighborhood_tree = {k: [] for k in range(1, num_neighborhoods + 1)}
    for parent in range(1, num_neighborhoods + 1):
        num_children = random.randint(1, max_children)
        children = random.sample(range(1, num_neighborhoods + 1), num_children)
        neighborhood_tree[parent] = [child for child in children if child != parent]

    # Generate bounds
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * 0.5)
        upper_bounds[k] = int(bound_tightness * total_impact * 1.5)

   # Adjust bounds for hierarchical constraints
    for parent, children in neighborhood_tree.items():
        if children:
            total_children_lower_bound = sum(lower_bounds[child] for child in children)
            total_children_upper_bound = sum(upper_bounds[child] for child in children)

            # Ensure bounds remain valid
            lower_bounds[parent] = max(lower_bounds[parent], int(0.8 * total_children_lower_bound))
            upper_bounds[parent] = max(lower_bounds[parent], min(upper_bounds[parent], int(1.2 * total_children_upper_bound)))

            # Add a safeguard to avoid lower bound exceeding upper bound
            if lower_bounds[parent] > upper_bounds[parent]:
                upper_bounds[parent] = lower_bounds[parent]


    return Knapsack(
        cost=costs,
        votes=votes,
        budget=budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )

##### 2.2.5.2 Run initial instances

In [ ]:
instances = []
for i in range(1):
    instance = generate_instance_multi_tiered(
        num_proposals= 3000,
        num_neighborhoods=40,
        budget=40000,
        cost_range= (20,100),
        vote_range= (10,100),
        impact_range= (10, 300),
        bound_tightness=0.3,
        max_children= 3,
        seed=42+i
    )
    instances.append(instance)

counter = 1
results_5 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=5)
    results_5.append(metrics)
    print(f"completed iteration {counter} " )
    counter += 1


evaluation_5 = evaluate_instances(results_5, group_by_method=False)

# Display evaluation results
display_evaluation_results(evaluation_5)

In [ ]:
metrics = []

# Define hyperparameter combinations (Adapt the grid while finding harder combinations) - Record combinations tried as comments
parameter_space = {
    "num_proposals": [3000],
    "num_neighborhoods": [40], # [10, 25, 50, 100]
    "budget_factor": [0.2], # 0.1,0.2
    "cost_range": [(20, 100)], #(20, 100), (10,50)
    "vote_range": [(10,100)], #(20, 100), (10,50)
    "impact_range": [(10,300)], # (10, 100), (10,200), (10,300),(50, 70)
    "bound_tightness": [0.3], # 0.1, 0,2, 0.3, 0.4, 0.15, 0.25,
    "max_children": [3], #2,3
}

# Create all combinations of hyperparameters
parameter_combinations = sorted(list(product(*parameter_space.values())))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for 1 iterations for each combination
        instance = generate_instance_multi_tiered(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            vote_range=param_dict["vote_range"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            max_children=param_dict["max_children"],
            seed=SEED + i
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=5)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)


In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget_factor", "cost_range", "vote_range", "bound_tightness", "impact_range", "max_children"]
hardness_analysis_df_5 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
hardness_analysis_df_5.sort_values(by='hardness_score', ascending = False).head(25)

#### Run Winner

In [ ]:
instances = []

for i in range(10): # Method takes too long so we only run 5 instances
    instance = generate_instance_multi_tiered(
        num_proposals= 3000,
        num_neighborhoods=40,
        budget=3000 * 60 * 0.2,
        cost_range= (20,100),
        vote_range= (10,100),
        impact_range=(10, 300),
        bound_tightness=0.3,
        max_children= 3,  # Max number of children per neighborhood
        seed=42+i
    )
    instances.append(instance)

counter = 1
results_5 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=5)
    results_5.append(metrics)
    print(f"completed iteration {counter} " )
    counter += 1


evaluation_5 = evaluate_instances(results_5, group_by_method=False)

# Display evaluation results
display_evaluation_results(evaluation_5)

In [ ]:
# Plot results of this instance generation for the 30 instances.
plot_metrics(results_5, evaluation_5)

## 2.2.6 Instance Generation Method: Correlated Costs with resource saturation (tighter bounds)

In [ ]:
def generate_instance_correlated(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    impact_range: Tuple[int, int],
    bound_tightness: float = 0.8,
    seed: int = None,
) -> Knapsack:
    """
    Generate a random instance of the knapsack problem with correlated parameters.

    Parameters:
        num_proposals (int): Total number of proposals.
        num_neighborhoods (int): Number of neighborhoods.
        budget (int): Budget for the instance.
        cost_range (Tuple[int, int]): Min and max range for proposal costs.
        impact_range (Tuple[int, int]): Min and max range for impacts.
        bound_tightness (float): Fraction of total neighborhood impact used for bounds.
        seed (int, optional): Seed for reproducibility. Defaults to None.

    Returns:
        Knapsack: A randomly generated knapsack instance.
    """
    # Set seed for reproducibility
    if seed is not None:
        random.seed(seed)

    # Generate costs
    costs = {i: random.randint(*cost_range) for i in range(1, num_proposals + 1)}

    # Generate votes correlated to costs (tighter range with noise)
    votes = {i: max(1, int(costs[i] * random.uniform(1.5, 1.7) + random.uniform(-5, 5))) for i in range(1, num_proposals + 1)}

    # Generate impacts correlated to votes (non-linear impacts with noise)
    impacts = {i: max(impact_range[0], min(impact_range[1], int(votes[i]**0.9 * random.uniform(0.6, 0.7) + random.uniform(-2, 2))))
               for i in range(1, num_proposals + 1)}

    # Assign proposals to neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    for proposal in range(1, num_proposals + 1):
        assigned_neighborhood = random.randint(1, num_neighborhoods)
        neighborhood_assignments[assigned_neighborhood].append(proposal)

    # Generate neighborhood bounds (tighter bounds)
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * 0.7)
        upper_bounds[k] = int(bound_tightness * total_impact * 1.2)

    # Create and return the knapsack instance
    return Knapsack(
        cost=costs,
        votes=votes,
        budget=budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )


In [ ]:
# Evaluation
instances = []
for i in range(1):
    instance = generate_instance_correlated(
        num_proposals=3000,
        num_neighborhoods=25,
        budget= 100000,
        cost_range=(10, 100),
        impact_range=(10, 100),
        bound_tightness=1,
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_6 = []
counter = 1
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=6)
    results_6.append(metrics)
    print(f"iteration {counter + 1} completed")
    counter += 1


evaluation_6 = evaluate_instances(results_6, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_6)

In [ ]:
metrics = []

# Define hyperparameter combinations
parameter_space = {
    "num_proposals": [3000],
    #"num_neighborhoods": [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114], # focusing in on the HARD instances
    "num_neighborhoods": [107], # focusing in on the HARD instances
    "budget_factor": [0.27],
    "cost_range": [(200, 1000)], #(10, 500), (500,1000), 100, 500), (20, 200)
    "impact_range": [(10, 500)],
    "bound_tightness": [0.21] # 0.1, 0.3, 0.4, 0.15, 0.2, 0.25,
}

# Create all combinations of hyperparameters
parameter_combinations = sorted(list(product(*parameter_space.values())))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for 1 iterations
        instance = generate_instance_correlated(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            seed=SEED + i
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=6)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget_factor", "cost_range", "bound_tightness", "impact_range"]
hardness_analysis_df_6 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
best_results_iteration_6 = hardness_analysis_df_6.sort_values(by='runtime_mean', ascending = False).head(15)
best_results_iteration_6['runtime_mean_in_minutes'] = best_results_iteration_6['runtime_mean'] / 60
best_results_iteration_6

In [ ]:
# Plot: Barplot of number of neighborhoods vs runtime mean
plt.figure(figsize=(16, 8))
plt.bar(best_results_iteration_6["num_neighborhoods"], best_results_iteration_6["runtime_mean_in_minutes"], color="skyblue", edgecolor="black")

# Customizing the plot
plt.title("Impact of Number of Neighborhoods on Runtime", fontsize=16)
plt.xlabel("Number of Neighborhoods", fontsize=14)
plt.ylabel("Runtime (Minutes)", fontsize=14)
plt.grid(axis='y', linestyle="--", alpha=1)
plt.xticks(np.arange(100, 115, step=1))
plt.yticks(np.arange(0, 30, step=1))

# Show the plot
plt.show()

<div class="alert alert-block alert-info" style="color: black;">
   This result is actually very interesting. The hardest instance is at 107 neighborhoods. This results shows us that difficulty does not scale linearly with neigborhood choice. We can see here that (all other parameters fixed), the runtime is significantly larger at 107 neighborhoods compared to its neighbors. This suggests that finding hard instances, is not a linear problem, as in "holding all else fixed, increasing x leads to a increase in difficulty". The only parameter for this which it tends to be unconditionally true is the number of proposals, which is simply the problem size.
<br> <br>
We also unfortunately have not been able to recover any methods to systematically discover hard instances, other than by trial-and-error.
</div>

##### Focusing on an easier instance -> 103 neighborhoods -> How does bound tightness in that neighborhood impact the runtime?

In [ ]:
metrics = []

# Define hyperparameter combinations
parameter_space = {
    "num_proposals": [3000],
    "num_neighborhoods": [103], # focusing in on the HARD instances
    "budget_factor": [0.27],
    "cost_range": [(200, 1000)], #(10, 500), (500,1000), 100, 500), (20, 200)
    "impact_range": [(10, 500)],
    "bound_tightness": [0.16, 0.17, 0.18, 0.19, 0.20, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26] # 0.1, 0.3, 0.4, 0.15, 0.2, 0.25,
}

# Create all combinations of hyperparameters
parameter_combinations = sorted(list(product(*parameter_space.values())))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for 1 iterations
        instance = generate_instance_correlated(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            seed=SEED + i
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=6)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget_factor", "cost_range", "bound_tightness", "impact_range"]
hardness_analysis_df_7 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
best_results_iteration_7 = hardness_analysis_df_7.sort_values(by='runtime_mean', ascending = False).head(15)
best_results_iteration_7['runtime_mean_in_minutes'] = best_results_iteration_7['runtime_mean'] / 60
best_results_iteration_7

In [ ]:
# Plot: Barplot of number of neighborhoods vs runtime mean
plt.figure(figsize=(12, 8))
plt.scatter(best_results_iteration_7["bound_tightness"], best_results_iteration_7["runtime_mean_in_minutes"], color="skyblue", edgecolor="black")

# Customizing the plot
plt.title("Impact of Bound Tightness on Runtime (num_neighborhoods = 103)", fontsize=16)
plt.xlabel("Bound Tightness", fontsize=14)
plt.ylabel("Runtime (Minutes)", fontsize=14)
plt.grid(axis='y', linestyle="--", alpha=1)

# Show the plot
plt.show()

<div class="alert alert-block alert-info" style="color: black;">
<b>How does bound tightness affect runtime (hardness), assumign all other parameters are fixed? <b>
</div>

<div class="alert alert-block alert-info" style="color: black;">
   For this example, we selected an "easy" instance from the previous (num_neighborhoods = 103) with a runtime of only 0.28 seconds, to see if we can tune this parameter separately. This result is also quite interesting. The hardest instance is at 0.21 bound tightness, in fact the same as that which we initially used. But the results suggests that the difficulty is fixed at a maximum and can rapidly decrease (to the right), but also seems to remain relatively consistent (to the left).
<br> <br>
Again, lots more exploration would be needed to verify that this result actually holds in the general case. But from initial observations, it seems to be the observed trend
</div>

##### Focusing on an easier instance -> 103 neighborhoods -> How does budget factor in that neighborhood impact the runtime?

In [ ]:
metrics = []

# Define hyperparameter combinations
parameter_space = {
    "num_proposals": [3000],
    "num_neighborhoods": [103], # focusing in on the HARD instances
    "budget_factor": [0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.30, 0.31],
    "cost_range": [(200, 1000)], #(10, 500), (500,1000), 100, 500), (20, 200)
    "impact_range": [(10, 500)],
    "bound_tightness": [0.21] # 0.1, 0.3, 0.4, 0.15, 0.2, 0.25,
}

# Create all combinations of hyperparameters
parameter_combinations = sorted(list(product(*parameter_space.values())))

# Generate instances and collect metrics for each combination
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for 1 iterations
        instance = generate_instance_correlated(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=int(param_dict["num_proposals"] * sum(param_dict["cost_range"]) / 2 * param_dict["budget_factor"]),
            cost_range=param_dict["cost_range"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            seed=SEED + i
        )

        solution = solve_mip(instance)
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=6)

        # Attach hyperparameters to metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget_factor", "cost_range", "bound_tightness", "impact_range"]
hardness_analysis_df_8 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
best_results_iteration_8 = hardness_analysis_df_8.sort_values(by='runtime_mean', ascending = False).head(15)
best_results_iteration_8['runtime_mean_in_minutes'] = best_results_iteration_8['runtime_mean'] / 60
best_results_iteration_8

In [ ]:
# Plot: Barplot of number of neighborhoods vs runtime mean
plt.figure(figsize=(12, 8))
plt.scatter(best_results_iteration_8["budget_factor"], best_results_iteration_8["runtime_mean_in_minutes"], color="skyblue", edgecolor="black")

# Customizing the plot
plt.title("Impact of Budget Factor on Runtime (num_neighborhoods = 103)", fontsize=16)
plt.xlabel("Budget Factor", fontsize=14)
plt.ylabel("Runtime (Minutes)", fontsize=14)
plt.grid(axis='y', linestyle="--", alpha=1)

# Show the plot
plt.show()

<div class="alert alert-block alert-info" style="color: black;">
It kind of appears as though budget factor has the inverse effect of budget tightness, though again, these are simply observations and not robust results.
</div>

#### Run Winner

In [ ]:
3000 * 600 * 0.27

In [ ]:
# Evaluation
instances = []
for i in range(5):
    instance = generate_instance_correlated(
        num_proposals=3000,
        num_neighborhoods=107,
        budget= 486000,
        cost_range=(200, 1000),
        impact_range=(10, 500),
        bound_tightness=0.21,
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_6 = []
counter = 1
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=6)
    results_6.append(metrics)
    print(f"iteration {counter} completed")
    counter += 1

# Evaluate and compare results grouped by method
evaluation_6 = evaluate_instances(results_6, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_6)

In [ ]:
# Plot results of this instance generation for the 30 instances.
plot_metrics(results_6, evaluation_6)

## 2.2.7 Instance Generation Method: Using a log-normal, multivariate distribution and bimodal to generate costs, votes and impacts

##### 2.2.7.1 Define the method

In [ ]:
import random
import numpy as np
from typing import Dict, List, Tuple
from dataclasses import dataclass

def generate_instance_skewed_multivariate(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    vote_range: Tuple[int, int],
    impact_range: Tuple[int, int],
    bound_tightness: float = 0.8,
    seed: int = None,
) -> 'Knapsack':
    """
    Generate a challenging random instance of the knapsack problem with more complexity.

    Parameters:
        num_proposals (int): Total number of proposals (fixed at 3000).
        num_neighborhoods (int): Number of neighborhoods.
        budget (int): Budget for the instance.
        cost_range (Tuple[int, int]): Min and max range for proposal costs.
        vote_range (Tuple[int, int]): Min and max range for proposal votes.
        impact_range (Tuple[int, int]): Min and max range for impacts.
        bound_tightness (float): Fraction of total neighborhood impact used for bounds.
        seed (int, optional): Seed for reproducibility. Defaults to None.

    Returns:
        Knapsack: A randomly generated knapsack instance with increased complexity.
    """
    # Set seed for reproducibility
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)


    cost_range = (cost_range[0], 200)
    vote_range = (vote_range[0], 500)
    impact_range = (impact_range[0], 100)

    # Generate costs with a log-normal distribution for increased skewness
    mu = np.log((cost_range[0] + cost_range[1]) / 2)
    sigma = 0.5  # Standard deviation
    costs = {
        i: max(cost_range[0], min(cost_range[1], int(np.random.lognormal(mu, sigma))))
        for i in range(1, num_proposals + 1)
    }

    # Generate votes using a bimodal distribution
    mid_vote = (vote_range[0] + vote_range[1]) / 2
    votes = {}
    for i in range(1, num_proposals + 1):
        if np.random.rand() < 0.5:
            votes[i] = max(vote_range[0], min(vote_range[1], int(np.random.normal(mid_vote - 50, 25))))
        else:
            votes[i] = max(vote_range[0], min(vote_range[1], int(np.random.normal(mid_vote + 50, 25))))

    # Generate impacts with a skewed distribution (Pareto)
    alpha = 1.5
    impacts = {
        i: int(np.random.pareto(alpha) * (impact_range[1] - impact_range[0]) + impact_range[0])
        for i in range(1, num_proposals + 1)
    }

    # Assign proposals to multiple neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    # First, distribute the proposals across neighborhoods.
    proposals_per_neighborhood = np.array_split(range(1, num_proposals + 1), num_neighborhoods)

    for i, neighborhood in enumerate(neighborhood_assignments.keys()):
        neighborhood_assignments[neighborhood] = list(proposals_per_neighborhood[i])

    # Ensure that the total number of proposals across all neighborhoods equals num_proposals
    total_proposals_in_neighborhoods = sum(len(proposals) for proposals in neighborhood_assignments.values())
    assert total_proposals_in_neighborhoods == num_proposals, \
        f"Mismatch: {total_proposals_in_neighborhoods} proposals assigned to neighborhoods, expected {num_proposals}"

    # Generate neighborhood bounds with increased variability
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * random.uniform(0.2, 0.5))  # Tighter lower bounds
        upper_bounds[k] = int(bound_tightness * total_impact * random.uniform(1.5, 2.0))  # Looser upper bounds

    # Increase budget complexity by reducing the budget slightly
    adjusted_budget = int(budget * random.uniform(0.75, 0.85))  # Further reduce budget to make it harder

    # Ensure that adjusted budget is reasonable
    adjusted_budget = max(adjusted_budget, cost_range[0] * num_proposals)

    # Create and return the knapsack instance
    return Knapsack(
        cost=costs,
        votes=votes,
        budget=adjusted_budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )


##### 2.2.7.2 Run initial instance

In [ ]:
# Evaluation
instances = []
for i in range(100):
    instance = generate_instance_skewed_multivariate(
        num_proposals=3000,
        num_neighborhoods=30,
        budget=35000,
        cost_range=(100, 200),
        vote_range=(20, 100),
        impact_range=(10, 100),
        bound_tightness=0.3,
        seed=SEED + i
)
    instances.append(instance)

results_7 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=7)
    results_7.append(metrics)

# Evaluate and compare results
evaluation_7 = evaluate_instances(results_7, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_7)

## 2.2.8 Instance Generation Method:

##### 2.2.8.1 Define the method

In [ ]:
def generate_instance_uniform_binomial_knapsack(
    num_proposals: int,
    num_neighborhoods: int,
    budget: int,
    cost_range: Tuple[int, int],
    vote_trials: int,
    vote_probability: float,
    impact_range: Tuple[int, int],
    bound_tightness: float = 0.8,
    seed: int = None,
) -> Knapsack:
    """
    Generate a Knapsack instance with uniform costs and binomial distribution for votes.

    Parameters:
        num_proposals (int): Total number of proposals.
        num_neighborhoods (int): Number of neighborhoods.
        budget (int): Budget for the instance.
        cost_range (Tuple[int, int]): Range for the uniform distribution of costs.
        vote_trials (int): Number of trials for the binomial distribution.
        vote_probability (float): Probability of success in binomial distribution.
        impact_range (Tuple[int, int]): Min and max range for impacts.
        bound_tightness (float): Fraction of total neighborhood impact used for bounds.
        seed (int, optional): Seed for reproducibility.

    Returns:
        Knapsack: A randomly generated knapsack instance.
    """
    if seed is not None:
        random.seed(seed)

    # Generate costs with uniform distribution
    costs = {i: random.randint(*cost_range) for i in range(1, num_proposals + 1)}

    # Generate votes using binomial distribution
    votes = {i: np.random.binomial(vote_trials, vote_probability) for i in range(1, num_proposals + 1)}

    # Generate impacts uniformly
    impacts = {i: random.randint(*impact_range) for i in range(1, num_proposals + 1)}

    # Assign proposals to neighborhoods
    neighborhood_assignments = {k: [] for k in range(1, num_neighborhoods + 1)}
    for proposal in range(1, num_proposals + 1):
        assigned_neighborhood = random.randint(1, num_neighborhoods)
        neighborhood_assignments[assigned_neighborhood].append(proposal)

    # Generate neighborhood bounds
    lower_bounds = {}
    upper_bounds = {}
    for k, proposals in neighborhood_assignments.items():
        total_impact = sum(impacts[j] for j in proposals)
        lower_bounds[k] = int(bound_tightness * total_impact * 0.5)
        upper_bounds[k] = int(bound_tightness * total_impact * 1.5)

    # Create and return the knapsack instance
    return Knapsack(
        cost=costs,
        votes=votes,
        budget=budget,
        neighborhood_assignments=neighborhood_assignments,
        impacts=impacts,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )


##### 2.2.8.2. Initial instances

In [ ]:
# Evaluation
instances = []
for i in range(2):
    instance = generate_instance_uniform_binomial_knapsack(
        num_proposals=3000,
        num_neighborhoods=50,
        budget=20000,
        cost_range=(10,100),
        vote_trials=100,
        vote_probability=0.8,
        impact_range=(10, 100),
        bound_tightness=0.1,
        seed=SEED + i  # Seed for reproducibility
)
instances.append(instance)

results_8 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=8)
    results_8.append(metrics)

# Evaluate and compare results
evaluation_8 = evaluate_instances(results_8, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_8)

##### 2.2.8.3 Hyperparameter tuning

In [ ]:
# Hyperparameter combinations
parameter_space = {
    "num_proposals": [3000],  # Number of proposals
    "num_neighborhoods": [10, 11, 12],  # Number of neighborhoods
    "budget": [20000],  # Budget
    "cost_range": [(10, 50)],  # Range for uniform distribution of costs
    "vote_trials": [100, 500],  # Number of trials for binomial distribution
    "vote_probability": [0.4],  # Probability for the binomial distribution
    "impact_range": [(10, 100)],  # Impact range for proposals
    "bound_tightness": [0.1],  # Bound tightness for neighborhoods
}

# Create all combinations of hyperparameters
parameter_combinations = list(product(*parameter_space.values()))

# Collect metrics for each combination
metrics = []
for params in parameter_combinations:
    param_dict = dict(zip(parameter_space.keys(), params))

    for i in range(1):  # Repeat for multiple runs per configuration
        # Generate knapsack instance with the current hyperparameter combination
        instance = generate_instance_uniform_binomial_knapsack(
            num_proposals=param_dict["num_proposals"],
            num_neighborhoods=param_dict["num_neighborhoods"],
            budget=param_dict["budget"],
            cost_range=param_dict["cost_range"],
            vote_trials=param_dict["vote_trials"],
            vote_probability=param_dict["vote_probability"],
            impact_range=param_dict["impact_range"],
            bound_tightness=param_dict["bound_tightness"],
            seed=42 + i,
        )

        # Solve the MIP problem for the generated instance
        solution = solve_mip(instance)

        # Collect metrics for the solution
        instance_metrics = collect_instance_metrics(instance, solution, generation_method=1)

        # Attach hyperparameters to the metrics
        instance_metrics.update(param_dict)
        metrics.append(instance_metrics)


##### 2.2.8.4 Analyze results

In [ ]:
# Analyze hyperparameter impact
hyperparameters = ["num_neighborhoods", "budget", "cost_range", "bound_tightness", "impact_range", "vote_trials", "vote_probability"]
hardness_analysis_df_9 = analyze_hyperparameter_impact(metrics, hyperparameters)
# Display the DataFrame
best_results_iteration_9 = hardness_analysis_df_9.sort_values(by='runtime_mean', ascending = False).head(15)
best_results_iteration_9['runtime_mean_in_minutes'] = best_results_iteration_9['runtime_mean'] / 60
best_results_iteration_9

##### 2.2.8.5 Run winner

In [ ]:
# Evaluation
instances = []
for i in range(100):
    instance = generate_instance_uniform_binomial_knapsack(
        num_proposals=3000,
        num_neighborhoods=11,
        budget=20000,
        cost_range=(10,100),
        vote_trials=100,
        vote_probability=0.4,
        impact_range=(10, 100),
        bound_tightness=0.1,
        seed=SEED + i  # Seed for reproducibility
)
    instances.append(instance)

results_8 = []
for instance in instances:
    solution = solve_mip(instance)
    metrics = collect_instance_metrics(instance, solution, generation_method=8)
    results_8.append(metrics)

# Evaluate and compare results
evaluation_8 = evaluate_instances(results_8, group_by_method=False)

# Display grouped evaluation results
display_evaluation_results(evaluation_8)

## 2.2.9 Instance Generation Method: Beta-Log Normal

##### 2.2.9.1 Define the method

##### 2.2.9.2 Run initial instance

##### 2.2.9.3 Hyperparameters

##### 2.2.9.4 Analyze results

##### 2.2.9.5 Run winner

##### 2.2.9.6 Plot results

## 2.2.10 Instance Generation Method:

##### 2.2.10.1 Define the method

##### 2.2.10.2 Run initial instance

##### 2.2.10.3 Hyperparameters

##### 2.2.10.4 Analyze results

##### 2.2.10.5 Run winner

##### 2.2.10.6 Plot results

## 3.1 Findings

Initial observations include that the combination of parameters that make instanced "hard" are not very obvious. Tiny changes in parameters can make the solution much easier or much harder for Gurobi to solve. Additionally, there do not appear to obvious trends, i.e the relationships between parameters and hardness are very difficult to identify. We were able to find hard instances mostly by trial and error, focusing in on around harder instances.
One observation to make is that hard instances do appear to be located around other locally hard instances, meaning that once a "hard" instance has been found say a 10x increase in runtime over other parameter combinations, then zooming in on this parameter space and looking at its neighborhood, is a seemingly good approach to try and identify harder instances, by slightly tweaking some of the parameters.

Unfortunately it was not entirely feasible to keep track of every possible combination of hyperparameters we used. Since the methodology was very much based on adaptive tuning. I.e. we did not want to create a grid of 1000 combinations since that would take too long to run, hence, the methodology used was along the lines of
1. Initialize a hyperparameter grid with approximately 100-200 combinations.
2. Identify the top combination and re-construct the hyperparameter grid with parameters changed by $\epsilon$ where $\epsilon$ is small
3. Iterate until "convergence" where convergence is not truly convergence but a runtime that we find acceptable
4. After "converged",

## 4.1 Statistical Tests

### 4.1.1 Comparing all methods

In [ ]:
import pandas as pd

def rank_methods(evaluation, all_results):
    """
    Rank methods based on their average runtime and include statistical significance, instance counts, and feasibility rate.

    Parameters:
        evaluation (dict): Evaluation results containing metrics for each method and pairwise comparisons.
        all_results (list): List of dictionaries containing runtime, node count, iteration count,
                            and generation method information for each run.

    Returns:
        Ranked DataFrame of methods with method names, average runtime, significance results, instance counts, and feasibility rate.
    """
    # Extract runtime for each method
    method_runtimes = {}
    method_names = {}  # Store method names for later inclusion
    for method, metrics in evaluation.items():
        if isinstance(method, int):
            method_runtimes[method] = metrics["Hardness Metrics"]["Average Runtime"]
            method_names[method] = metrics.get("Method Name", f"Method {method}")  # Default to "Method X" if no name

    # Count the number of runs (instances) and calculate feasibility rate for each generation method
    instance_counts = {}
    feasibility_counts = {}  # Store counts of feasible runs for each method

    for result in all_results:
        gen_method = result.get("generation_method")
        feasible = result.get("feasible", False)
        if gen_method is not None:
            instance_counts[gen_method] = instance_counts.get(gen_method, 0) + 1
            if feasible:
                feasibility_counts[gen_method] = feasibility_counts.get(gen_method, 0) + 1

    # Calculate feasibility rate
    feasibility_rate = {
        method: (feasibility_counts.get(method, 0) / instance_counts[method]) * 100
        if method in instance_counts else 0
        for method in instance_counts
    }

    # Create a DataFrame
    df = pd.DataFrame.from_dict(method_runtimes, orient="index", columns=["Average Runtime"])
    df["Method"] = df.index.map(method_names)  # Map method names to the DataFrame
    df["Instances Run"] = df.index.map(instance_counts).fillna(0).astype(int)  # Map instance counts to the DataFrame
    df["Feasibility Rate (%)"] = df.index.map(feasibility_rate).fillna(0)
    df["Feasibility Rate (%)"] = round(df["Feasibility Rate (%)"], 2)

    # Rank methods by runtime (descending order since higher runtime is ranked higher)
    df["Rank"] = df["Average Runtime"].rank(method="min", ascending=False).astype(int)

    # Format runtimes for better readability
    #df["Average Runtime (min)"] = round(df["Average Runtime"] / 60, 2)
    df["Average Runtime"] = round(df["Average Runtime"], 2)

    # Rename runtime column
    df.rename({'Average Runtime': 'Average Runtime (sec)'}, axis=1, inplace=True)

    # Add significance information from pairwise comparisons
    pairwise_comparisons = evaluation.get("Pairwise Statistical Comparisons", {})
    significance_results = {}

    for method in method_runtimes.keys():
        significant_methods = []
        for other_method, stats in pairwise_comparisons.get("runtime", {}).items():
            if str(method) in other_method.split(" vs "):
                t_stat = stats["T-Statistic"]
                p_val = stats["P-Value"]
                if p_val is not None and p_val < 0.05:
                    if method == int(other_method.split(" vs ")[0]):
                        significant_methods.append(int(other_method.split(" vs ")[1]))
                    else:
                        significant_methods.append(int(other_method.split(" vs ")[0]))
        significance_results[method] = ", ".join(map(str, significant_methods)) if significant_methods else "None"

    # Add significance results to the DataFrame
    df["Statistically Significant Compared To"] = df.index.map(significance_results)

    # Sort by rank
    df = df.sort_values("Rank")

    # Order columns
    df = df[['Rank', 'Method', 'Average Runtime (sec)', 'Instances Run', 'Feasibility Rate (%)', 'Statistically Significant Compared To']].reset_index(drop=True)

    return df


In [ ]:
# Assuming you have results_1, results_2, ..., results_10
all_results = results_1 + results_2 + results_3 + results_4 + results_5 + results_6 + results_7 + results_8

# Evaluate all results, grouped by generation method
evaluation = evaluate_instances(all_results, group_by_method=True)

# Display the evaluation results
display_evaluation_results(evaluation)

In [ ]:
rank_methods(evaluation, all_results)

# END